# Milestone2 — definitive Colab GPU worker
Floating memories • 1080p24 • 12 samples • ProRes 422 + 16-bit depth. Select a GPU runtime and Run all. Authorize the Google account with write access to the linked job folder. This notebook uses Drive folder IDs rather than ambiguous mounted paths. Run one worker per job.

In [ ]:
from google.colab import auth
auth.authenticate_user()
from google.auth import default
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
from pathlib import Path
import os, json, hashlib, shutil, io, time, subprocess, tarfile
creds,_=default()
import httplib2, google_auth_httplib2
drive_api=build('drive','v3',http=google_auth_httplib2.AuthorizedHttp(creds,http=httplib2.Http(timeout=120)),cache_discovery=False)
JOB_ID='1qYdX-8ia0THTt0KFI5qzKBjt7xNJjA13'
JOB_JSON_ID='1fztia838tPSsw3c-_j4zpkmc2-sBr2Wq'
INPUT_ID='14Bq_F6WTdkIAb6DLd7ljcR72fj-wdFwV'
folder=drive_api.files().get(fileId=JOB_ID,fields='id,name,capabilities(canAddChildren)',supportsAllDrives=True).execute()
assert folder['capabilities']['canAddChildren'], 'Authorize an account with write access to the job folder'
print('Verified Drive job:',folder['name'],folder['id'])
def children(parent):
    q=f"'{parent}' in parents and trashed=false"
    token=None; result=[]
    while True:
        r=drive_api.files().list(q=q,fields='nextPageToken,files(id,name,size,md5Checksum,mimeType)',pageSize=1000,pageToken=token,supportsAllDrives=True,includeItemsFromAllDrives=True).execute()
        result.extend(r.get('files',[]))
        token=r.get('nextPageToken')
        if not token:return result
def lookup(parent,name):
    matches=[x for x in children(parent) if x['name']==name]
    assert len(matches)<=1, f'Duplicate Drive name: {name}'
    return matches[0] if matches else None
def folder_id(parent,name):
    f=lookup(parent,name)
    if f:
        assert f['mimeType']=='application/vnd.google-apps.folder'
        return f['id']
    return drive_api.files().create(body={'name':name,'parents':[parent],'mimeType':'application/vnd.google-apps.folder'},fields='id',supportsAllDrives=True).execute()['id']
def file_hash(path,algorithm='md5'):
    with path.open('rb') as f:return hashlib.file_digest(f,algorithm).hexdigest()
def index_files(parent):
    result={}
    for f in children(parent):
        if f['name'] in result:raise RuntimeError('Duplicate Drive name: '+f['name'])
        result[f['name']]=f
    return result
def download(fid,dst,expected=None):
    expected=expected or drive_api.files().get(fileId=fid,fields='id,name,size,md5Checksum',supportsAllDrives=True).execute()
    def matches(path):
        return path.is_file() and path.stat().st_size==int(expected['size']) and file_hash(path)==expected['md5Checksum']
    if matches(dst):return False
    dst.parent.mkdir(parents=True,exist_ok=True)
    tmp=dst.with_suffix(dst.suffix+'.downloading')
    with tmp.open('wb') as f:
        loader=MediaIoBaseDownload(f,drive_api.files().get_media(fileId=fid,supportsAllDrives=True),chunksize=32*1024*1024)
        done=False
        while not done:
            status,done=loader.next_chunk(num_retries=5)
    if not matches(tmp):raise RuntimeError("Downloaded checksum mismatch: "+dst.name)
    os.replace(tmp,dst)
    return True
def upload(src,parent,existing=None):
    with src.open('rb') as f: md5=hashlib.file_digest(f,'md5').hexdigest()
    if existing and existing.get('md5Checksum')==md5 and int(existing.get('size',-1))==src.stat().st_size:return existing
    # Refuse conflicting existing content instead of overwriting committed media.
    if existing:raise RuntimeError('Conflicting Drive file: '+src.name)
    media=MediaFileUpload(str(src),resumable=True,chunksize=8*1024*1024)
    args=dict(media_body=media,fields='id,name,size,md5Checksum',supportsAllDrives=True)
    if existing:req=drive_api.files().update(fileId=existing['id'],**args)
    else:req=drive_api.files().create(body={'name':src.name,'parents':[parent]},**args)
    result=None
    while result is None: _,result=req.next_chunk(num_retries=5)
    assert result['md5Checksum']==md5 and int(result['size'])==src.stat().st_size, 'Upload verification failed'
    return result
cfg=json.loads(drive_api.files().get_media(fileId=JOB_JSON_ID,supportsAllDrives=True).execute())
assert cfg['version']==2 and cfg['scene']=='floating' and cfg['count']==2420
print('Job:',cfg['scene'],cfg['count'],'frames')


In [ ]:
import pathlib, subprocess, urllib.request, hashlib, tarfile, json, shutil
from pathlib import Path
subprocess.run(['nvidia-smi'], check=True)
base='https://download.blender.org/release/Blender5.0/'
name='blender-5.0.1-linux-x64.tar.xz'
archive=Path('/content')/name
exe=Path('/content/blender-5.0.1-linux-x64/blender')
if not exe.is_file():
    urllib.request.urlretrieve(base+name,archive)
    expected=next(line.split()[0] for line in urllib.request.urlopen(base+'blender-5.0.1.sha256').read().decode().splitlines() if line.endswith(name))
    with archive.open('rb') as f: assert hashlib.file_digest(f,'sha256').hexdigest()==expected, 'Blender checksum mismatch'
    subprocess.run(['tar','-xJf',str(archive),'-C','/content'],check=True)

import shutil
assert shutil.which('ffmpeg') and shutil.which('ffprobe'), 'FFmpeg required'


In [19]:
work=Path('/content/milestone2-worker')/cfg['sha256'][:16]
work.mkdir(parents=True,exist_ok=True)
bundle=work/'input.tar'
def digest(path):
    with path.open('rb') as f:return hashlib.file_digest(f,'sha256').hexdigest()
if not bundle.exists() or digest(bundle)!=cfg['sha256']:
    print('Downloading verified geometry package (2.32 GB)',flush=True)
    download(INPUT_ID,bundle)
assert digest(bundle)==cfg['sha256'], 'Input checksum mismatch'
# Refuse extraction/render changes while a worker in this runtime is active.
import fcntl
from contextlib import contextmanager
@contextmanager
def worker_lock():
    with (work/'worker.lock').open('a') as lock:
        try:fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
        except BlockingIOError:raise RuntimeError('A worker is already running in this runtime')
        try:yield
        finally:fcntl.flock(lock,fcntl.LOCK_UN)
def patch_renderer(source):
    old='scene.cycles.denoiser="OPTIX" if a.device=="OPTIX" else "OPENIMAGEDENOISE"'
    new='scene.cycles.denoiser="OPENIMAGEDENOISE"\n    scene.cycles.denoising_use_gpu=True'
    if new in source and old not in source:return source
    if source.count(old)!=1:raise RuntimeError('Unexpected renderer revision')
    return source.replace(old,new,1)
def prepare_cache():
    # Reuse an extracted cache only when all expected archive members exist.
    stamp=work/'extracted.json'
    with tarfile.open(bundle) as t:
        members=t.getmembers()
        ready=stamp.is_file() and stamp.read_text()==cfg['sha256']
        ready=ready and all((work/m.name).is_file() and ((work/m.name).stat().st_size==m.size or m.name=='blender/render_cache.py') for m in members if m.isfile())
        if not ready:
            stamp.unlink(missing_ok=True)
            t.extractall(work,filter='data')
            tmp=stamp.with_suffix('.tmp');tmp.write_text(cfg['sha256']);os.replace(tmp,stamp)
    renderer=work/'blender/render_cache.py'
    source=renderer.read_text()
    patched=patch_renderer(source)
    if patched!=source:
        tmp=renderer.with_suffix('.tmp');tmp.write_text(patched);os.replace(tmp,renderer)
    print('Verified cache prepared; OptiX + GPU OIDN',flush=True)
with worker_lock():prepare_cache()

meta=json.loads((work/'cache/manifest.json').read_text())
assert meta['count']==cfg['count'] and meta['settings']['buildHash']==cfg['buildHash']
out=work/'render.prores';out.mkdir(exist_ok=True)
remote=folder_id(JOB_ID,'render.prores')
chunks_id=folder_id(remote,'chunks')
depth_id=folder_id(remote,'depth')
signature=out/'job-settings.json'
signature.write_text(json.dumps(cfg,sort_keys=True))
old=lookup(remote,signature.name)
if old:
    assert json.loads(drive_api.files().get_media(fileId=old['id'],supportsAllDrives=True).execute())==cfg,'Changed job settings'
else:upload(signature,remote)
remote_chunks=index_files(chunks_id)
remote_depth=index_files(depth_id)
with worker_lock():
    for name,mark in sorted(remote_chunks.items()):
        if not name.endswith('.json'):continue
        dst=out/'chunks'/name
        download(mark['id'],dst,mark)
        d=json.loads(dst.read_text())
        mov=dst.with_suffix('.mov')
        download(remote_chunks[mov.name]['id'],mov,remote_chunks[mov.name])
        for i in range(d['start'],d['start']+d['count']):
            p=out/'depth'/f'frame-{i:06d}.png'
            download(remote_depth[p.name]['id'],p,remote_depth[p.name])
print('Verified cache and restored committed chunks',flush=True)


Verified cache prepared; OptiX + GPU OIDN
Verified cache and restored committed chunks


In [20]:
# Optional authenticated monitor; disable if you do not want a public tunnel.
ENABLE_MONITOR = True
"""Optional read-only Colab monitor. No remote command execution."""
import base64
import hashlib
import hmac
import json
import os
import re
import secrets
import shutil
import subprocess
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from collections import deque
from datetime import datetime, timezone

class Monitor:
    def __init__(self, output, total, password=None, publish=None):
        self.output=Path(output)
        self.total=total
        self.password=password or secrets.token_urlsafe(24)
        self.publish=publish
        self.stop=threading.Event()
        self.logs=deque(maxlen=100)
        self.lock=threading.Lock()
        self.url=None
        self.ssh=None
        self.server=None
        self.thread=None
        self.frame=0
        self.uploaded=0
        self.uploading=None
        self.timings=deque(maxlen=30)
        self.started=time.monotonic()
    def log(self,line):
        with self.lock:
            # Only accept renderer lines, never authentication/tunnel output.
            self.logs.append(line.rstrip()[:1000])
            m=re.search(r'Finished (\d+) seconds ([\d.]+)',line)
            if m:
                self.frame=max(self.frame,int(m[1]))
                self.timings.append(float(m[2]))
    def status(self):
        depth=len(list((self.output/'depth').glob('frame-*.png')))
        committed=0
        chunks=0
        for p in (self.output/'chunks').glob('chunk-*.json'):
            try:
                d=json.loads(p.read_text())
                committed+=int(d['count']);chunks+=1
            except (OSError,ValueError,KeyError):pass
        with self.lock:
            frame=max(depth,self.frame)
            avg=sum(self.timings)/len(self.timings) if self.timings else None
        return dict(rendered=frame,total=self.total,committed=committed,chunks=chunks,
                    percent=round(frame/self.total*100,2),seconds_per_frame=avg,
                    eta_seconds=round(max(0,self.total-frame)*avg) if avg else None,
                    uploaded=self.uploaded,uploading=self.uploading,
                    complete=(self.output/'media.json').is_file(),
                    updated=datetime.now(timezone.utc).isoformat())
    def gpu(self):
        try:
            p=subprocess.run(['nvidia-smi','--query-gpu=name,utilization.gpu,memory.used,memory.total,temperature.gpu','--format=csv,noheader,nounits'],capture_output=True,text=True,timeout=5,check=True)
            return [dict(zip(('name','utilization_percent','vram_used_mib','vram_total_mib','temperature_c'),[s.strip() for s in l.split(',')])) for l in p.stdout.splitlines()]
        except Exception:return []
    def system(self):
        d=shutil.disk_usage(self.output.parent)
        mem={}
        try:
            for line in Path('/proc/meminfo').read_text().splitlines():
                if line.startswith(('MemTotal:','MemAvailable:')):
                    k,v=line.split(':',1);mem[k]=int(v.split()[0])*1024
        except OSError:pass
        return dict(disk_free_bytes=d.free,disk_total_bytes=d.total,memory=mem,load_average=os.getloadavg())
    def start(self):
        if self.server:return self
        mon=self
        class Handler(BaseHTTPRequestHandler):
            def log_message(self,*args):pass
            def do_GET(self):
                expected='Basic '+base64.b64encode(('monitor:'+mon.password).encode()).decode()
                if not hmac.compare_digest(self.headers.get('Authorization',''),expected):
                    self.send_response(401);self.send_header('WWW-Authenticate','Basic realm="Render monitor"');self.end_headers();return
                route=self.path.split('?',1)[0]
                if route=='/api/status':data=mon.status()
                elif route=='/api/gpu':data=mon.gpu()
                elif route=='/api/system':data=mon.system()
                elif route=='/api/uploads':data=dict(uploaded=mon.uploaded,uploading=mon.uploading,total=mon.total)
                elif route=='/api/logs':
                    with mon.lock:data=list(mon.logs)
                elif route=='/':
                    data=('<!doctype html><meta charset="utf-8"><meta name="viewport" content="width=device-width"><title>Render monitor</title><style>body{font:16px system-ui;max-width:720px;margin:2em auto;padding:1em;background:#111827;color:#eee}pre{white-space:pre-wrap;background:#1f2937;padding:1em;border-radius:8px}progress{width:100%}</style><h1>Colab render monitor</h1><progress id="bar" max="100"></progress><pre id="status"></pre><pre id="gpu"></pre><pre id="system"></pre><pre id="logs"></pre><script>async function update(){for(let name of ["status","gpu","system","logs"]){try{let r=await fetch("/api/"+name);if(!r.ok)throw Error(r.status);let j=await r.json();document.getElementById(name).textContent=JSON.stringify(j,null,2);if(name==="status")document.getElementById("bar").value=j.percent}catch(e){document.getElementById(name).textContent="Unavailable: "+e}}}update();setInterval(update,5000)</script>').encode()
                    return self.reply(200,data,'text/html; charset=utf-8')
                else:return self.reply(404,b'Not found','text/plain')
                self.reply(200,json.dumps(data).encode(),'application/json')
            def reply(self,code,body,typ):
                self.send_response(code);self.send_header('Content-Type',typ);self.send_header('Content-Length',str(len(body)));self.send_header('Cache-Control','no-store');self.send_header('X-Content-Type-Options','nosniff');self.end_headers();self.wfile.write(body)
        self.server=ThreadingHTTPServer(('127.0.0.1',0),Handler)
        threading.Thread(target=self.server.serve_forever,daemon=True).start()
        self.thread=threading.Thread(target=self.tunnel,daemon=True)
        self.thread.start()
        return self
    def tunnel(self):
        # SSH host keys are accepted on first use, not disabled on subsequent connections.
        port=self.server.server_address[1]
        # A dedicated, ephemeral SSH key avoids password prompts on reconnect.
        key=self.output.parent/'monitor-ssh-key'
        if not key.exists():
            subprocess.run(['ssh-keygen','-q','-t','ed25519','-N','','-f',str(key)],check=True)
        while not self.stop.is_set():
            try:
                self.ssh=subprocess.Popen(['ssh','-tt','-i',str(key),'-o','IdentitiesOnly=yes','-o','BatchMode=yes','-p','443','-o','StrictHostKeyChecking=accept-new','-o','ServerAliveInterval=30','-o','ServerAliveCountMax=3','-o','ConnectTimeout=15','-R',f'0:127.0.0.1:{port}','free.pinggy.io'],stdin=subprocess.DEVNULL,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
                for line in self.ssh.stdout:
                    if self.stop.is_set():break
                    m=re.search(r'https://[a-zA-Z0-9.-]+\.pinggy(?:-free)?\.(?:link|io|run)(?=[^a-zA-Z0-9.-]|$)',line)
                    if m and m[0]!=self.url:
                        self.url=m[0]
                        print('[MONITOR] '+self.url,flush=True)
                        if self.publish:
                            try:self.publish(self.url)
                            except Exception:print('[MONITOR] URL publication failed',flush=True)
                self.ssh.wait(timeout=15)
            except Exception:
                if self.ssh and self.ssh.poll() is None:self.ssh.terminate()
            self.url=None
            self.stop.wait(10)
    def close(self):
        self.stop.set()
        if self.ssh and self.ssh.poll() is None:self.ssh.terminate()
        if self.server:self.server.shutdown();self.server.server_close()

def publish_monitor_url(url):
    # Separate Drive client because httplib2 is not thread safe.
    client=build('drive','v3',credentials=creds,cache_discovery=False)
    name='monitor-endpoint.json'
    data=json.dumps({'url':url,'updated':datetime.now(timezone.utc).isoformat(),'auth':'basic','username':'monitor'}).encode()
    from googleapiclient.http import MediaIoBaseUpload
    q=f"'{JOB_ID}' in parents and trashed=false and name='{name}'"
    files=client.files().list(q=q,fields='files(id),nextPageToken',supportsAllDrives=True,includeItemsFromAllDrives=True).execute().get('files',[])
    if len(files)>1:raise RuntimeError('Duplicate monitor endpoint metadata')
    media=MediaIoBaseUpload(io.BytesIO(data),mimetype='application/json')
    if files:client.files().update(fileId=files[0]['id'],media_body=media,supportsAllDrives=True).execute(num_retries=3)
    else:client.files().create(body={'name':name,'parents':[JOB_ID]},media_body=media,supportsAllDrives=True).execute(num_retries=3)

def start_monitor():
    global render_monitor
    existing=globals().get('render_monitor')
    if existing and existing.server and not existing.stop.is_set():return existing
    from getpass import getpass
    password=getpass('Set monitor password (at least 16 characters; not saved): ')
    if len(password)<16:raise ValueError('Monitor password too short')
    render_monitor=Monitor(out,cfg['count'],password=password,publish=publish_monitor_url).start()
    print('Monitor username: monitor',flush=True)
    return render_monitor

# Upload each verified chunk, with its completion marker last.
def sync_chunks():
    global remote_chunks, remote_depth
    marks=sorted((out/'chunks').glob('chunk-*.json'))
    if not any(m.name not in remote_chunks for m in marks):return
    remote_chunks=index_files(chunks_id)
    remote_depth=index_files(depth_id)
    for mark in marks:
        if mark.name in remote_chunks:
            assert file_hash(mark)==remote_chunks[mark.name]['md5Checksum'], 'Conflicting committed chunk'
            continue
        d=json.loads(mark.read_text())
        if globals().get('render_monitor'):render_monitor.uploading=d['start']
        for i in range(d['start'],d['start']+d['count']):
            p=out/'depth'/f'frame-{i:06d}.png'
            remote_depth[p.name]=upload(p,depth_id,remote_depth.get(p.name))
        mov=mark.with_suffix('.mov')
        remote_chunks[mov.name]=upload(mov,chunks_id,remote_chunks.get(mov.name))
        remote_chunks[mark.name]=upload(mark,chunks_id)
        if globals().get('render_monitor'):
            render_monitor.uploaded+=d['count']
            render_monitor.uploading=None
        print('Verified Drive chunk:',d['start'],'frames:',d['count'],flush=True)
cmd=[str(exe),'-b','--factory-startup','--python-exit-code','1','-P',str(work/'blender/render_cache.py'),'--','--cache',str(work/'cache'),'--output',str(out),'--format','PNG','--passes','beauty','--prores','--device','OPTIX','--ffmpeg',shutil.which('ffmpeg'),'--ffprobe',shutil.which('ffprobe')]
for k in ('width','height','samples','bounces','noise','min_samples','diffuse','glossy','chunk_size','depth_far'):
    cmd+=['--'+k.replace('_','-'),str(cfg[k])]
def retryable_upload_error(exc):
    from googleapiclient.errors import HttpError
    import socket, ssl, httplib2
    if isinstance(exc,HttpError):
        status=exc.resp.status
        return status in (408,429,500,502,503,504) or (status==403 and any(reason in str(exc) for reason in ('rateLimitExceeded','userRateLimitExceeded','backendError')))
    return isinstance(exc,(TimeoutError,ConnectionError,socket.timeout,ssl.SSLError,httplib2.HttpLib2Error))

def drain_uploads(attempt, stop_event, producer_done):
    # A failing Drive request must never terminate the renderer.
    delay=5
    while not stop_event.is_set():
        try:
            final=producer_done.is_set()
            attempt()
            delay=5
            if final:return
        except Exception as exc:
            print('[DRIVE] Upload deferred: '+type(exc).__name__,flush=True)
            if not retryable_upload_error(exc):
                print('[DRIVE] Permanent upload failure; local output retained. Rerun after fixing access/conflict.',flush=True)
                raise
            delay=min(300,max(10,delay*2))
        stop_event.wait(delay)

def run_worker():
    global remote_chunks, remote_depth
    remote_chunks=index_files(chunks_id)
    remote_depth=index_files(depth_id)
    if globals().get('render_monitor'):
        render_monitor.uploaded=sum(json.loads(m.read_text())['count'] for m in (out/'chunks').glob('chunk-*.json') if m.name in remote_chunks)
    # Verify the OptiX compute device before launching; do not silently fall back.
    subprocess.run([str(exe),'-b','--factory-startup','--python-exit-code','1','--python-expr',
        "import bpy; p=bpy.context.preferences.addons['cycles'].preferences; p.compute_device_type='OPTIX'; p.refresh_devices(); assert any(d.type=='OPTIX' for d in p.devices), 'No OptiX GPU detected'"],check=True)
    import threading
    print('Launching OptiX + GPU OIDN at',cfg['samples'],'samples',flush=True)
    proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    def stream_logs():
        for line in proc.stdout:
            print('BLENDER:',line.rstrip(),flush=True)
            if ENABLE_MONITOR and globals().get('render_monitor'):render_monitor.log(line)
    log_thread=threading.Thread(target=stream_logs,daemon=True)
    log_thread.start()
    producer_done=threading.Event()
    stop_upload=threading.Event()
    upload_errors=[]
    def uploader():
        try:drain_uploads(sync_chunks,stop_upload,producer_done)
        except Exception as exc:upload_errors.append(exc)
    upload_thread=threading.Thread(target=uploader,daemon=True)
    upload_thread.start()
    try:
        while proc.poll() is None:
            time.sleep(2)
        log_thread.join(timeout=10)
        producer_done.set()
        while upload_thread.is_alive():
            upload_thread.join(timeout=5)
    finally:
        if proc.poll() is None:
            proc.terminate()
            proc.wait()
        producer_done.set()
        stop_upload.set()
        upload_thread.join(timeout=10)
        log_thread.join(timeout=10)
    assert proc.returncode==0, 'Render interrupted; rerun to resume'
    if upload_errors:raise RuntimeError('Rendering complete; upload incomplete') from upload_errors[0]
    assert not upload_thread.is_alive(), 'Upload still active; rerun after it exits'
    # Final metadata is published only after every local chunk has been uploaded.
    for name in ('depth-metadata.json','media.json'):
        delay=5
        while True:
            try:
                upload(out/name,remote,lookup(remote,name))
                break
            except Exception as exc:
                if not retryable_upload_error(exc):raise
                print('[DRIVE] Metadata upload deferred',flush=True)
                time.sleep(delay)
                delay=min(300,delay*2)
    print('Complete: https://drive.google.com/drive/folders/'+remote)

with worker_lock():
    if ENABLE_MONITOR:
        try:start_monitor()
        except Exception as e:
          print(f"Monitor startup failed: {e}", flush=True)
    run_worker()


Set monitor password (at least 16 characters; not saved): ··········
Monitor startup failed; continuing render
Launching OptiX + GPU OIDN at 12 samples
BLENDER: 00:00.732  cycles           | WARNING HIPEW initialization failed: Error opening HIP dynamic library
BLENDER: [cycles] GPU: Tesla T4
BLENDER: /content/milestone2-worker/10233c239b2295f8/blender/render_cache.py:94: DeprecationWarning: 'World.use_nodes' is expected to be removed in Blender 6.0
BLENDER:   scene.world.use_nodes=True
BLENDER: /content/milestone2-worker/10233c239b2295f8/blender/render_cache.py:122: DeprecationWarning: 'Material.use_nodes' is expected to be removed in Blender 6.0
BLENDER:   mat=bpy.data.materials.new("Material-%d"%m["id"]);mat.use_nodes=True
BLENDER: [cycles] Mesh datablocks: 289 for 720 objects
BLENDER: [cycles] 720 meshes; 27 materials; 2420 frames
BLENDER: [cycles] Replay starting at frame 480
BLENDER: [cycles] Resume 481
BLENDER: [cycles] Resume 482
BLENDER: [cycles] Resume 483
BLENDER: [cycles] R

KeyboardInterrupt: 